In [1]:
# Identificaciones y par?metros personalizados confirmados por el grupo.
C1 = 1023092749
C2 = 1023092750
S = C1 + C2
nivel_confianza = {0: 0.90, 1: 0.95, 2: 0.99}[S % 3]
tau_ms = 180 + 10 * (S % 9)
epsilon = 0.01 * (1 + S % 3)
g = 1.4 + 0.1 * (S % 7)
t1_kb = 80 + 10 * (S % 5)
t2_kb = 400 + 25 * (S % 5)
pi = (4 + S % 6) / 1000
k_max = 4 + S % 5
r = S % 6
N = 10 + S % 11
print(f"S={S}, confianza={nivel_confianza:.0%}, tau={tau_ms} ms, epsilon={epsilon}, "
      f"g={g:.1f}, t1={t1_kb} KB, t2={t2_kb} KB, pi={pi}, k_max={k_max}, r={r}, N={N}")

S=2046185499, confianza=90%, tau=210 ms, epsilon=0.01, g=1.5, t1=120 KB, t2=500 KB, pi=0.007, k_max=8, r=3, N=16


# Partes A y B ? auditor?a, reg?menes y an?lisis

El r?gimen estable une [0,668) y [838,1200]. Consulte `documentos/calculso/actualizacion_A_B.md` para decisiones, f?rmulas y limitaciones.

In [2]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
if not (ROOT / "src").exists():
    raise FileNotFoundError("Ejecute el notebook desde la ra?z del proyecto o notebooks/")
sys.path.insert(0, str(ROOT / "src"))
from auditoria import audit
from regimenes import main as segmentar
from llegadas_intentos import main as analizar_b
from descriptivos import main as resumen_a3

## A1. Auditor?a
Se convierten los tama?os indicados y se excluyen filas inv?lidas siguiendo la tolerancia confirmada. Se conservan los motivos en las salidas.

In [3]:
solicitudes, conteos_auditoria = audit()

Filas originales: 34072; identificadores consolidados: 33971; válidas: 33737; excluidas: 234
duplicado exacto eliminado: 101
filas excluidas por datos de latencia inválidos o ausentes: 234
gpu_ms: ausente o contradictorio: 33
latencia_ms: inferior a red más GPU, fuera del redondeo: 67
latencia_ms: tiempo negativo: 101
red_ms: ausente o contradictorio: 34
tam_kb: convertido de bytes a KB: 50


## A2. Cambios de r?gimen
El an?lisis conserva BIC para uno a cinco segmentos y eval?a sensibilidad. El r?gimen elegido une los tramos [0,668) y [838,1200].

In [4]:
segmentar()

ancho=1s segmentos=[(0, 668, 16353), (668, 838, 8781), (838, 1200, 8603)] BIC=[-157630.6 -158319.5 -160866.9 -160854.2 -160843.2]


ancho=5s segmentos=[(0, 665, 16276), (665, 840, 8920), (840, 1200, 8541)] BIC=[-266227.5 -266914.2 -269408.5 -269398.7 -269389.9]
ancho=10s segmentos=[(0, 670, 16463), (670, 840, 8733), (840, 1200, 8541)] BIC=[-312997.6 -313669.8 -316172.7 -316165.4 -316157.6]
ancho=15s segmentos=[(0, 660, 16162), (660, 720, 2873), (720, 840, 6161), (840, 1200, 8541)] BIC=[-340356.3 -341029.4 -343446.2 -343447.  -343441.1]
ancho=30s segmentos=[(0, 660, 16162), (660, 720, 2873), (720, 840, 6161), (840, 1200, 8541)] BIC=[-387126.4 -387800.9 -390219.1 -390221.3 -390216.7]


Comparación de tramos: [{'prueba': 'KS tiempos entre llegadas', 'estadistico': np.float64(0.01420774466048258), 'p_valor': np.float64(0.20328224077138246)}, {'prueba': 'KS latencia', 'estadistico': np.float64(0.007467960713526456), 'p_valor': np.float64(0.9092863516753398)}, {'prueba': 'Mann-Whitney latencia', 'estadistico': np.float64(70673721.0), 'p_valor': np.float64(0.5402309621242771)}]


## A3. Estad?stica descriptiva
Por versi?n y r?gimen: varianza y desviaci?n muestrales con denominador n?1; asimetr?a y exceso de curtosis corregidos por sesgo.

In [5]:
resumen_a3()

Resumen A3: 16 combinaciones por variable, versión y régimen


## B. Llegadas, fallos y reintentos
Los intervalos entre llegadas se calculan por separado en los dos tramos estables. Se registran las salidas y los supuestos de las pruebas y simulaciones.

In [6]:
analizar_b(S)

{
  "B1": {
    "n": 24956,
    "duration_s": 1030,
    "rate_s": 24.229126213592235,
    "confidence": 0.9,
    "rate_interval": [
      23.977402453844743,
      24.482929810762933
    ],
    "mean_gap_s": 0.04127318666346076,
    "model_mean_gap_s": 0.04127263984612919,
    "model_mean_gap_interval_s": [
      0.040844784824747175,
      0.0417059354917593
    ],
    "numero_gaps": 24954,
    "max_gap_s": 0.48490000000015243,
    "ks_stat": 0.005051614741657384,
    "ks_bootstrap_p": 0.3516209476309227,
    "bootstrap_replicas": 400,
    "count_window_s": 10,
    "ventanas_completas": 102,
    "exposicion_excluida_s": 10,
    "fano": 0.8161327143077004,
    "dispersion_p": 0.17737514304197916,
    "ties": 42
  },
  "B2": [
    {
      "regimen": "estable",
      "solicitudes": 24956,
      "bloques": 249,
      "solicitudes_restantes": 56,
      "proporcion": 0.15298926109953517,
      "media_bloques": 15.301204819277109,
      "media_binomial": 15.298926109953518,
      "varianza_b